# 05. Feature Engineering V3 — 성분 기능군 Count

**목적**  
성분을 기능군으로 매핑하고 제품별 기능 성분 개수를 생성합니다.

**입력**  
`data/interim/전성분_표준화_최종.csv`

**출력**  
`data/interim/V3_성분기능군_개수.csv`

> 저장된 전처리 데이터만 사용하며 외부 요청은 발생하지 않습니다.


In [ ]:
from pathlib import Path

# Jupyter와 Colab 모두 저장소 루트에서 실행합니다.
def find_project_root(start=Path.cwd()):
    for path in [start.resolve(), *start.resolve().parents]:
        if (path / "data").is_dir() and (path / "notebooks").is_dir():
            return path
    raise FileNotFoundError("저장소를 clone한 뒤 해당 폴더 안에서 실행하세요.")

PROJECT_ROOT = find_project_root()

DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"

for directory in [DATA_RAW_DIR, DATA_INTERIM_DIR, DATA_PROCESSED_DIR, REPORTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)


# KDMS V3 — 기능군 기반 성분 Feature Engineering

이 노트북은 `전성분_표준화_최종.csv`의 `canonical_name`을 기능군에 다중 매핑하고,
제품별 기능군 Count와 Ratio 피처를 생성합니다.

## 최종 생성 파일은 3개뿐입니다.

1. `V3_성분기능군_개수.csv`  
   모델 실험용 최종 파일. Count와 Ratio를 한 파일에 저장합니다.

2. `V3_성분기능군_매핑표.csv`  
   어떤 성분이 어떤 기능군으로 분류됐는지 확인하는 재현성·근거 파일입니다.

3. `V3_미매칭성분_검토목록.csv`  
   여러 제품에서 등장하지만 기능군에 아직 분류되지 않은 성분만 모은 검토 파일입니다.

모델 실험에서는 `V3_성분기능군_개수.csv`만 사용하면 됩니다.


## 1. 경로 설정


In [ ]:
from pathlib import Path
from collections import defaultdict
import re
import unicodedata

import numpy as np
import pandas as pd

INPUT_PATH = DATA_INTERIM_DIR / "전성분_표준화_최종.csv"
OUTPUT_DIR = DATA_INTERIM_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("입력 파일:", INPUT_PATH)
print("출력 폴더:", OUTPUT_DIR)
print("입력 파일 존재 여부:", INPUT_PATH.exists())


## 2. 데이터 불러오기

- `canonical_name`을 성분 식별 기준으로 사용합니다.
- 같은 제품에 같은 성분이 여러 번 있으면 한 번만 집계합니다.
- 이후 `product_group_id`가 추가된 파일을 사용하면 해당 컬럼을 자동으로 우선 사용합니다.


In [2]:

df = pd.read_csv(INPUT_PATH, encoding='utf-8-sig')

required = {'product_id', 'product_name', 'canonical_name'}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"필수 컬럼 누락: {sorted(missing)}")

ID_COL = 'product_group_id' if 'product_group_id' in df.columns else 'product_id'

def normalize_name(value):
    if pd.isna(value):
        return ''
    value = unicodedata.normalize('NFKC', str(value))
    value = re.sub(r'\s+', '', value).strip().lower()
    return value

df['canonical_name'] = df['canonical_name'].astype('string').str.strip()
df = df[df['canonical_name'].notna() & df['canonical_name'].ne('')].copy()
df['ingredient_norm'] = df['canonical_name'].map(normalize_name)

base = (
    df[[ID_COL, 'product_name', 'canonical_name', 'ingredient_norm']]
    .drop_duplicates([ID_COL, 'ingredient_norm'])
    .reset_index(drop=True)
)

print("사용 ID:", ID_COL)
print("제품 수:", base[ID_COL].nunique())
print("고유 성분 수:", base['ingredient_norm'].nunique())
print("제품-성분 관계 수:", len(base))

display(base.head())


사용 ID: product_id
제품 수: 231
고유 성분 수: 1292
제품-성분 관계 수: 10282


,product_id,product_name,canonical_name,ingredient_norm
0,A000000260257,제로이드 수딩 크림,정제수,정제수
1,A000000260257,제로이드 수딩 크림,글리세린,글리세린
2,A000000260257,제로이드 수딩 크림,프로판다이올,프로판다이올
3,A000000260257,제로이드 수딩 크림,카프릴릭/카프릭트라이글리세라이드,카프릴릭/카프릭트라이글리세라이드
4,A000000260257,제로이드 수딩 크림,솔비탄스테아레이트,솔비탄스테아레이트


## 3. 기능군 정의

한 성분은 여러 기능군에 동시에 포함될 수 있습니다.

예:

- 판테놀 → 보습 + 장벽 + 진정
- 다이메티콘 → 밀폐 + 실리콘
- 살리실릭애씨드 → 활성 + 산/각질

기능군 Ratio의 합이 1을 넘을 수 있으며 이는 정상입니다.


In [ ]:
GROUP_LABELS = {
    'humectant': '보습',
    'emollient': '유연',
    'occlusive': '밀폐',
    'barrier': '장벽',
    'soothing': '진정',
    'active': '활성',
    'fragrance': '향료',
    'acid_exfoliant': '산_각질',
    'silicone': '실리콘',
    'peptide': '펩타이드',
    'botanical_extract': '식물추출물',
    'oil_butter': '오일_버터',
}
GROUPS = list(GROUP_LABELS)


In [ ]:
EXACT_BY_GROUP = {
    'humectant': [
        '글리세린', '부틸렌글라이콜', '프로판다이올', '펜틸렌글라이콜',
        '다이프로필렌글라이콜', '메틸프로판다이올', '1,2-헥산다이올',
        '카프릴릴글라이콜', '소듐하이알루로네이트',
        '하이알루로닉애씨드', '하이드롤라이즈드하이알루로닉애씨드',
        '소듐하이알루로네이트크로스폴리머',
        '소듐아세틸레이티드하이알루로네이트',
        '하이드록시프로필트라이모늄하이알루로네이트',
        '하이드롤라이즈드소듐하이알루로네이트',
        '포타슘하이알루로네이트', '베타인', '트레할로오스',
        '글루코오스', '자일리톨', '만니톨', '솔비톨',
        '소듐피씨에이', '판테놀', '베타-글루칸', '엑토인',
        '글리세릴글루코사이드', '하이드롤라이즈드콜라겐',
        '알지닌', '글라이신',
    ],
    'emollient': [
        '카프릴릭/카프릭트라이글리세라이드', '스쿠알란',
        '세테아릴알코올', '베헤닐알코올', '세틸알코올',
        '스테아릴알코올', '글리세릴스테아레이트',
        '세틸에틸헥사노에이트', '다이카프릴릴카보네이트',
        '다이카프릴릴에터', '트라이에틸헥사노인',
        '에틸헥실팔미테이트', '아이소노닐아이소노나노에이트',
        '하이드로제네이티드폴리데센',
        '하이드로제네이티드폴리아이소부텐',
        '호호바에스터', '펜타에리스리틸테트라에틸헥사노에이트',
    ],
    'occlusive': [
        '페트롤라툼', '미네랄오일', '파라핀', '세레신',
        '오조케라이트', '비즈왁스', '라놀린', '시어버터',
        '스쿠알란', '다이메티콘', '다이메티콘올',
        '비닐다이메티콘', '메틸트라이메티콘',
        '카프릴릴메티콘', '하이드로제네이티드폴리데센',
        '하이드로제네이티드폴리아이소부텐',
    ],
    'barrier': [
        '세라마이드엔피', '세라마이드에이피', '세라마이드이오피',
        '세라마이드엔에스', '세라마이드에이에스',
        '콜레스테롤', '피토스핑고신',
        '하이드로제네이티드레시틴', '레시틴',
        '판테놀', '스쿠알란', '팔미틱애씨드',
        '스테아릭애씨드', '리놀레익애씨드',
        '리놀레닉애씨드', '베타-글루칸', '엑토인',
    ],
    'soothing': [
        '판테놀', '알란토인', '마데카소사이드',
        '병풀추출물', '병풀잎추출물',
        '아시아티코사이드', '아시아틱애씨드',
        '마데카식애씨드', '다이포타슘글리시리제이트',
        '베타-글루칸', '엑토인', '비스아보롤', '칼라민',
    ],
    'active': [
        '나이아신아마이드', '아데노신', '토코페롤',
        '토코페릴아세테이트', '글루타티온', '알부틴',
        '트라넥사믹애씨드', '바쿠치올', '레티놀',
        '레틴알', '레티닐팔미테이트', '아스코빅애씨드',
        '3-O-에틸아스코빅애씨드', '아스코빌글루코사이드',
        '소듐아스코빌포스페이트',
        '마그네슘아스코빌포스페이트',
        '유비퀴논', '카페인', '소듐디엔에이',
    ],
    'fragrance': [
        '향료', '리모넨', '리날룰', '시트랄',
        '시트로넬올', '제라니올', '유제놀',
        '아이소유제놀', '쿠마린', '파네솔',
        '벤질살리실레이트', '벤질벤조에이트',
        '헥실신남알', '알파-아이소메틸아이오논',
        '아니스알코올',
    ],
    'acid_exfoliant': [
        '살리실릭애씨드', '카프릴로일살리실릭애씨드',
        '글라이콜릭애씨드', '락틱애씨드',
        '만델릭애씨드', '글루코노락톤',
        '락토바이오닉애씨드', '아젤라익애씨드',
        '석시닉애씨드',
    ],
    'silicone': [
        '다이메티콘', '다이메티콘올', '비닐다이메티콘',
        '메틸트라이메티콘', '카프릴릴메티콘',
        '사이클로펜타실록세인', '사이클로헥사실록세인',
        '폴리메틸실세스퀴옥세인',
    ],
    'peptide': [
        '아세틸헥사펩타이드-8',
        '카퍼트라이펩타이드-1',
        '트라이펩타이드-1',
    ],
    'botanical_extract': [],
    'oil_butter': [
        '시어버터', '해바라기씨오일', '올리브오일',
    ],
}


In [ ]:
KEYWORD_RULES = {
    'humectant': [
        r'하이알루', r'히알루', r'소듐피씨에이',
        r'폴리글루타믹', r'사카라이드아이소머레이트',
        r'콜라겐$', r'베타-?글루칸',
        r'트레할로오스', r'자일리톨',
        r'만니톨', r'솔비톨',
    ],
    'emollient': [
        r'트라이글리세라이드', r'스쿠알[란렌]',
        r'폴리데센', r'폴리아이소부텐',
        r'아이소노나노에이트', r'에틸헥사노에이트',
        r'팔미테이트$', r'카보네이트$',
        r'다이카프릴릴에터', r'호호바에스터', r'버터$',
    ],
    'occlusive': [
        r'페트롤라툼', r'미네랄오일', r'파라핀',
        r'왁스', r'세레신', r'오조케라이트',
        r'라놀린', r'버터$', r'다이메티콘',
        r'실록세인', r'실세스퀴옥세인',
        r'폴리데센', r'폴리아이소부텐',
    ],
    'barrier': [
        r'세라마이드', r'콜레스테롤',
        r'피토스핑고신', r'스핑고',
        r'레시틴', r'인지질',
        r'리놀레익애씨드', r'리놀레닉애씨드',
    ],
    'soothing': [
        r'병풀', r'마데카소사이드',
        r'아시아티코사이드', r'아시아틱애씨드',
        r'마데카식애씨드', r'글리시리제이트',
        r'감초', r'알로에', r'캐모마일',
        r'마트리카리아', r'카모밀라',
        r'녹차', r'어성초', r'약모밀',
        r'쑥', r'아르테미시아',
        r'귀리', r'오트', r'비스아보롤',
    ],
    'active': [
        r'나이아신아마이드', r'아데노신',
        r'토코페롤', r'토코페릴',
        r'레티놀', r'레틴알', r'레티닐',
        r'바쿠치올', r'아스코빅', r'아스코빌',
        r'알부틴', r'트라넥사믹', r'글루타티온',
        r'유비퀴논', r'카페인',
        r'펩타이드', r'소듐디엔에이',
        r'폴리데옥시리보',
        r'살리실릭애씨드', r'글라이콜릭애씨드',
        r'락틱애씨드', r'만델릭애씨드',
        r'글루코노락톤', r'락토바이오닉애씨드',
        r'아젤라익애씨드',
    ],
    'fragrance': [
        r'^향료$', r'리모넨', r'리날룰',
        r'시트랄$', r'시트로넬올',
        r'제라니올', r'유제놀',
        r'아이소유제놀', r'쿠마린',
        r'파네솔', r'벤질살리실레이트',
        r'벤질벤조에이트', r'헥실신남알',
        r'아이소메틸아이오논', r'아니스알코올',
    ],
    'acid_exfoliant': [
        r'살리실릭애씨드', r'카프릴로일살리실릭',
        r'글라이콜릭애씨드', r'락틱애씨드',
        r'만델릭애씨드', r'글루코노락톤',
        r'락토바이오닉애씨드',
        r'아젤라익애씨드', r'석시닉애씨드',
    ],
    'silicone': [
        r'다이메티콘', r'메티콘',
        r'실록세인', r'실세스퀴옥세인',
        r'실리콘',
    ],
    'peptide': [
        r'펩타이드', r'트라이펩타이드',
        r'테트라펩타이드', r'펜타펩타이드',
        r'헥사펩타이드', r'올리고펩타이드',
        r'폴리펩타이드',
    ],
    'botanical_extract': [
        r'추출물$', r'잎추출물$', r'뿌리추출물$',
        r'꽃추출물$', r'열매추출물$',
        r'씨추출물$', r'껍질추출물$',
        r'줄기추출물$', r'전초추출물$', r'수액$',
    ],
    'oil_butter': [
        r'오일$', r'씨오일$', r'열매오일$',
        r'껍질오일$', r'버터$',
    ],
}


In [3]:
exact_lookup = defaultdict(set)
for group, names in EXACT_BY_GROUP.items():
    for name in names:
        exact_lookup[normalize_name(name)].add(group)

compiled_rules = {
    group: [re.compile(pattern) for pattern in patterns]
    for group, patterns in KEYWORD_RULES.items()
}

def map_ingredient(name):
    norm = normalize_name(name)
    groups = set(exact_lookup.get(norm, set()))
    evidence = []

    if groups:
        evidence.append('exact')

    for group, patterns in compiled_rules.items():
        if any(pattern.search(norm) for pattern in patterns):
            groups.add(group)
            evidence.append(f'keyword:{group}')

    return groups, '|'.join(evidence)


## 4. 성분 기능군 매핑

`V3_성분기능군_매핑표.csv`는 연구 방법 재현과 매핑 검토를 위한 파일입니다.

`V3_미매칭성분_검토목록.csv`에는 최소 5개 제품 이상에서 등장했지만
기능군에 매핑되지 않은 성분만 저장합니다.


In [4]:

ingredient_stats = (
    base.groupby(['canonical_name', 'ingredient_norm'], as_index=False)
    .agg(product_count=(ID_COL, 'nunique'))
)

mapped_result = ingredient_stats['canonical_name'].map(map_ingredient)
ingredient_stats['group_set'] = mapped_result.map(lambda x: x[0])
ingredient_stats['mapping_evidence'] = mapped_result.map(lambda x: x[1])
ingredient_stats['functional_groups'] = ingredient_stats['group_set'].map(
    lambda values: '|'.join(sorted(values))
)
ingredient_stats['mapped'] = ingredient_stats['group_set'].map(bool)

mapping_output = (
    ingredient_stats[
        [
            'canonical_name',
            'product_count',
            'functional_groups',
            'mapping_evidence',
            'mapped',
        ]
    ]
    .sort_values(
        ['product_count', 'canonical_name'],
        ascending=[False, True],
    )
)

MAPPING_PATH = OUTPUT_DIR / 'V3_성분기능군_매핑표.csv'
mapping_output.to_csv(
    MAPPING_PATH,
    index=False,
    encoding='utf-8-sig',
)

# 실질적으로 검토 가치가 있는 반복 등장 미분류 성분만 저장
unmapped_review = mapping_output[
    (~mapping_output['mapped'])
    & (mapping_output['product_count'] >= 5)
].copy()

UNMAPPED_PATH = OUTPUT_DIR / 'V3_미매칭성분_검토목록.csv'
unmapped_review.to_csv(
    UNMAPPED_PATH,
    index=False,
    encoding='utf-8-sig',
)

print("성분 매핑 수:",
      f"{mapping_output['mapped'].sum()} / {len(mapping_output)}")
print("검토 대상 미분류 성분 수:", len(unmapped_review))

display(mapping_output.head(20))
display(unmapped_review.head(20))


성분 매핑 수: 682 / 1292
검토 대상 미분류 성분 수: 170


,canonical_name,product_count,functional_groups,mapping_evidence,mapped
830,정제수,224,,,False
71,글리세린,221,humectant,exact,True
0,"1,2-헥산다이올",200,humectant,exact,True
391,부틸렌글라이콜,192,humectant,exact,True
736,에틸헥실글리세린,162,,,False
1010,판테놀,138,barrier|humectant|soothing,exact,True
892,카프릴릭/카프릭트라이글리세라이드,137,emollient,exact|keyword:emollient,True
967,토코페롤,135,active,exact|keyword:active,True
528,소듐하이알루로네이트,132,humectant,exact|keyword:humectant,True
90,나이아신아마이드,119,active,exact|keyword:active,True


,canonical_name,product_count,functional_groups,mapping_evidence,mapped
830,정제수,224,,,False
736,에틸헥실글리세린,162,,,False
824,잔탄검,108,,,False
126,다이소듐이디티에이,101,,,False
668,아크릴레이트/C10-30알킬아크릴레이트크로스폴리머,93,,,False
872,카보머,88,,,False
991,트로메타민,87,,,False
540,솔비탄올리베이트,81,,,False
700,암모늄아크릴로일다이메틸타우레이트/브이피코폴리머,78,,,False
1195,하이드록시에틸아크릴레이트/소듐아크릴로일다이메틸타우레이트코폴리머,78,,,False


## 5. 제품별 기능군 Feature 생성

최종 모델 입력 파일에는 다음 컬럼만 저장합니다.

- 제품 ID
- 제품명
- 제품의 전체 고유 성분 수
- 기능군별 Count
- 기능군별 Ratio

Count 실험을 할 때는 `v3_count_` 컬럼만 선택하고,
Ratio 실험을 할 때는 `v3_ratio_` 컬럼만 선택하면 됩니다.

따라서 Count와 Ratio를 별도 CSV로 나눌 필요가 없습니다.


In [ ]:
ingredient_to_groups = dict(
    zip(ingredient_stats['ingredient_norm'], ingredient_stats['group_set'])
)

product_ingredient = base.copy()
product_ingredient['group_set'] = (
    product_ingredient['ingredient_norm']
    .map(lambda x: ingredient_to_groups.get(x, set()))
)

product_meta = (
    product_ingredient.groupby(ID_COL, as_index=False)
    .agg(
        product_name=('product_name', 'first'),
        v3_total_ingredient_count=('ingredient_norm', 'nunique'),
    )
)

mapped_long = (
    product_ingredient[
        product_ingredient['group_set'].map(bool)
    ][
        [
            ID_COL,
            'ingredient_norm',
            'group_set',
        ]
    ]
    .explode('group_set')
    .rename(columns={'group_set': 'functional_group'})
)

count_wide = (
    mapped_long
    .groupby([ID_COL, 'functional_group'])['ingredient_norm']
    .nunique()
    .unstack(fill_value=0)
    .reindex(columns=GROUPS, fill_value=0)
)

count_wide.columns = [
    f'v3_count_{group}' for group in count_wide.columns
]
count_wide = count_wide.reset_index()

features = product_meta.merge(
    count_wide,
    on=ID_COL,
    how='left',
)

count_columns = [f'v3_count_{group}' for group in GROUPS]

for column in count_columns:
    features[column] = features[column].fillna(0).astype('int16')


In [ ]:
for group in GROUPS:
    features[f'v3_ratio_{group}'] = (
        features[f'v3_count_{group}']
        / features['v3_total_ingredient_count']
    ).fillna(0.0)

ratio_columns = [f'v3_ratio_{group}' for group in GROUPS]

features = features[
    [
        ID_COL,
        'product_name',
        'v3_total_ingredient_count',
    ]
    + count_columns
    + ratio_columns
].copy()

FEATURE_PATH = OUTPUT_DIR / 'V3_성분기능군_개수.csv'
features.to_csv(
    FEATURE_PATH,
    index=False,
    encoding='utf-8-sig',
)

assert features[ID_COL].is_unique
assert features.isna().sum().sum() == 0
assert len(features) == base[ID_COL].nunique()

print("V3 최종 Feature 생성 완료")
print("제품 수:", len(features))
print("Count 컬럼 수:", len(count_columns))
print("Ratio 컬럼 수:", len(ratio_columns))

display(features.head())


## 6. 최종 파일 확인

실제 모델링 담당자에게는 `V3_성분기능군_개수.csv` 하나만 전달하면 됩니다.

- Count 실험: `v3_count_`로 시작하는 컬럼
- Ratio 실험: `v3_ratio_`로 시작하는 컬럼
- Count + Ratio 실험: 두 종류 모두 사용

나머지 두 파일은 모델 입력이 아니라 연구 재현성과 분류 검토용입니다.


In [ ]:

print("생성된 최종 파일")
print("1.", FEATURE_PATH.name, "- 모델 실험용")
print("2.", MAPPING_PATH.name, "- 기능군 매핑 근거")
print("3.", UNMAPPED_PATH.name, "- 반복 등장 미분류 성분 검토")

print("\n저장 위치:", OUTPUT_DIR)
